In [2]:
!pip install sentence-transformers faiss-cpu google-generativeai

In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

import google.generativeai as genai

from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
GOOGLE_API_KEY = userdata.get(
    "Research++"
)

genai.configure(
    api_key=GOOGLE_API_KEY
)

llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print(
    "Gemini configured successfully."
)

Gemini configured successfully.


In [11]:
from google.colab import files

uploaded = files.upload()
uploaded = files.upload()

Saving researchforge_faiss.index to researchforge_faiss (1).index


Saving papers.json to papers (1).json


In [14]:
df = pd.read_json(
    "papers.json"
)

faiss_index = faiss.read_index(
    "researchforge_faiss.index"
)

print(
    f"Loaded {len(df)} papers."
)

Loaded 100 papers.


In [15]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print(
    "Embedding model ready."
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [16]:
def retrieve_papers(
    user_query,
    top_k=8
):

    query_embedding = embedding_model.encode(
        [user_query]
    )

    distances, indices = faiss_index.search(
        np.array(
            query_embedding
        ).astype("float32"),
        k=top_k
    )

    retrieved_papers = []

    for idx in indices[0]:

        retrieved_papers.append({

            "title":
            df.iloc[idx]["title"],

            "abstract":
            df.iloc[idx]["abstract"],

            "published":
            df.iloc[idx]["published"]

        })

    return retrieved_papers

In [17]:
def compare_methods(
    user_query,
    retrieved_papers
):

    evidence_context = ""

    for paper in retrieved_papers:

        evidence_context += f"""

Title:
{paper['title']}

Abstract:
{paper['abstract']}

"""

    prompt = f"""

You are an expert AI research analyst.

User Request:

{user_query}

Retrieved Scientific Evidence:

{evidence_context}

Task:

Perform a structured scientific comparison.

Return:

1. Compared methods

2. Architecture differences

3. Dataset usage

4. Evaluation metrics

5. Strengths

6. Limitations

7. Generalization analysis

8. Final evidence-grounded conclusion

Use retrieved evidence only.

"""

    response = llm.generate_content(
        prompt
    )

    return response.text

In [18]:
query = """
Compare transformer-based and CNN-based
multimodal deepfake detection methods.
"""

papers = retrieve_papers(
    query
)

comparison = compare_methods(
    query,
    papers
)

print(
    comparison
)

Here is a structured scientific comparison of transformer-based and CNN-based multimodal deepfake detection methods, based solely on the retrieved evidence:

### 1. Compared methods

*   **Transformer-based:**
    *   **AV-LMMDetect:** A supervised fine-tuned (SFT) large multimodal model built on Qwen 2.5 Omni, designed for audio-visual deepfake detection.
    *   **DeepFake-Adapter:** A parameter-efficient tuning approach for deepfake detection utilizing large pre-trained Vision Transformers (ViTs).
    *   Generally referred to as "Transformer architectures," "large multimodal models (LMMs)," and methods leveraging "large pre-trained ViTs."
*   **CNN-based:**
    *   "Convolutional neural network (CNN)" models that typically focus on "direct image processing."
    *   Often categorized under "small, task-specific models" or methods "merely focusing on low-level forgery patterns."
    *   One study also tested a "CNN model" with facial landmarks for deepfake detection.
    *   Implici